In [64]:
import pandas as pd
from google import genai
from dotenv import load_dotenv
import os
import json
import time

# Limpieza inicial

Se tienen los datos crudos de los resultados de la encuesta, ahora se hará una limpieza inicial sobre ellos. 

In [17]:
df = pd.read_csv('../data/dataset_encuesta.csv', encoding='latin-1', sep=';')

df.head()

,Id,Start time,Completion time,Email,Name,¿Aceptas participar voluntariamente en esta encuesta?\n,¿Ha recibido alguna vez un SMS que le pareció sospechoso?,¿De quién decía ser el mensaje?,¿A qué categoría pertenece la estafa que intentaron hacerle?,"Escriba el texto del primer SMS sospechoso que recibió, tal como apareció en su teléfono.","Escriba el texto de un SMS legítimo que haya recibido recientemente, tal como apareció en su teléfono.\n \nEjemplos de mensajes que puede compartir:\n Notificación de transacción bancaria Promoción de T",¿De qué tipo es el mensaje legítimo?
0,1,24/03/2026 9:33,24/03/2026 9:35,anonymous,NaN,"Si, acepto participar",Si,Banco o entidad financiera,"Financiero (préstamos, inversiones, criptomon...","Se autorizó un cargo por Q8,500 en BAC. Si no ...",¡Felicidades! Has ganado un Smartphone 5G de T...,"Promoción o aviso de operadora (Tigo, Claro, M..."
1,2,24/03/2026 9:33,24/03/2026 9:57,anonymous,NaN,"Si, acepto participar",Si,Empresa de paquetería o envíos,"Paquetes y envíos (maletas, importación, entr...","Hemos intentado entregar tu paquete dos veces,...",Eres un cliente DIAMANTE! Claro Premia tu Fide...,"Promoción o aviso de operadora (Tigo, Claro, M..."
2,3,24/03/2026 18:25,24/03/2026 18:26,anonymous,NaN,"Si, acepto participar",Si,Empresa de paquetería o envíos,"Paquetes y envíos (maletas, importación, entr...",NaN,Confirmación de entrega de pedido,Confirmación de compra o entrega
3,4,24/03/2026 18:24,24/03/2026 18:29,anonymous,NaN,"Si, acepto participar",Si,Empresa de paquetería o envíos,"Paquetes y envíos (maletas, importación, entr...",Hola correo central!! te han enviado un paquet...,Confirmar pedido,Código de verificación (2FA)
4,5,24/03/2026 18:31,24/03/2026 18:36,anonymous,NaN,"Si, acepto participar",Si,Empresa de paquetería o envíos,"Paquetes y envíos (maletas, importación, entr...","Intentamos entregar tu paquete, pero no pudimo...",Bicredit: Te recordamos que tienes un cambio d...,"Notificación bancaria (movimiento, saldo, alerta)"


In [18]:
df_spam = df[['Escriba el texto del primer SMS sospechoso que recibió, tal como apareció en su teléfono.', '¿A qué categoría pertenece la estafa que intentaron hacerle?']]
df_ham = df[['Escriba el texto de un SMS legítimo que haya recibido recientemente, tal como apareció en su teléfono.\n \nEjemplos de mensajes que puede compartir:\n  Notificación de transacción bancaria Promoción de T', '¿De qué tipo es el mensaje legítimo?']]

df_spam = df_spam.dropna()
df_ham = df_ham.dropna()

In [19]:
df_spam = df_spam.rename(columns={'Escriba el texto del primer SMS sospechoso que recibió, tal como apareció en su teléfono.': 'Mensaje', '¿A qué categoría pertenece la estafa que intentaron hacerle?': 'Categoria'})
df_ham = df_ham.rename(columns={'Escriba el texto de un SMS legítimo que haya recibido recientemente, tal como apareció en su teléfono.\n \nEjemplos de mensajes que puede compartir:\n  Notificación de transacción bancaria Promoción de T': 'Mensaje', '¿De qué tipo es el mensaje legítimo?': 'Categoria'})

In [20]:
df_spam['tipo'] = 'Spam'
df_ham['tipo'] = 'Ham'

In [21]:
df_spam.head()

,Mensaje,Categoria,tipo
0,"Se autorizó un cargo por Q8,500 en BAC. Si no ...","Financiero (préstamos, inversiones, criptomon...",Spam
1,"Hemos intentado entregar tu paquete dos veces,...","Paquetes y envíos (maletas, importación, entr...",Spam
3,Hola correo central!! te han enviado un paquet...,"Paquetes y envíos (maletas, importación, entr...",Spam
4,"Intentamos entregar tu paquete, pero no pudimo...","Paquetes y envíos (maletas, importación, entr...",Spam
6,Guatex: Su paquete ha llegado al almacén pero ...,"Paquetes y envíos (maletas, importación, entr...",Spam


In [23]:
df_ham.head()

,Mensaje,Categoria,tipo
0,¡Felicidades! Has ganado un Smartphone 5G de T...,"Promoción o aviso de operadora (Tigo, Claro, M...",Ham
1,Eres un cliente DIAMANTE! Claro Premia tu Fide...,"Promoción o aviso de operadora (Tigo, Claro, M...",Ham
2,Confirmación de entrega de pedido,Confirmación de compra o entrega,Ham
3,Confirmar pedido,Código de verificación (2FA),Ham
4,Bicredit: Te recordamos que tienes un cambio d...,"Notificación bancaria (movimiento, saldo, alerta)",Ham


In [25]:
df_final =  pd.concat([df_spam, df_ham], ignore_index=True)

In [31]:
df_final.head()

,Mensaje,Categoria,tipo
0,"Se autorizó un cargo por Q8,500 en BAC. Si no ...","Financiero (préstamos, inversiones, criptomon...",Spam
1,"Hemos intentado entregar tu paquete dos veces,...","Paquetes y envíos (maletas, importación, entr...",Spam
2,Hola correo central!! te han enviado un paquet...,"Paquetes y envíos (maletas, importación, entr...",Spam
3,"Intentamos entregar tu paquete, pero no pudimo...","Paquetes y envíos (maletas, importación, entr...",Spam
4,Guatex: Su paquete ha llegado al almacén pero ...,"Paquetes y envíos (maletas, importación, entr...",Spam


In [32]:
df_final.to_csv('../data/dataset_tranformed.csv')

In [28]:
api_key = os.environ.get('GEMINI_API_KEY')

In [68]:
client = genai.Client(api_key=api_key)

all_variations = []

for idx, row in df_final.iterrows():
    prompt = f"""
Eres un experto en ciberseguridad guatemalteco analizando SMS fraudulentos.

Dado este mensaje SMS original:
- Texto: "{row['Mensaje']}"
- Tipo: "{row['tipo']}"
- Categoría: "{row['Categoria']}"

Genera exactamente 8 variaciones sintéticas que:
- Mantengan la misma etiqueta ({row['tipo']}) y categoría ({row['Categoria']})
- Usen instituciones guatemaltecas reales (EMETRA, SAT, Banrural, BAM, IGSS, Tigo, Claro, Renap, Migración)
- Usen español guatemalteco natural (mezcla de vos/usted según contexto)

{"Para variaciones SPAM (fraudulentos): Variar nivel de urgencia (mayoría sutil), incluir URLs falsas (.icu, .xyz, .info, .tk, bit.ly), variar montos entre Q150 y Q5,000" if row['tipo'] == 'spam' else "Para variaciones HAM (legítimos): NO incluir URLs sospechosas, mantener tono formal e institucional, variar montos, fechas y detalles menores"}

Responde ÚNICAMENTE con un array JSON válido con exactamente 8 objetos, sin texto adicional:
[
  {{
    "texto": "texto completo del SMS",
    "categoria": "{row['Categoria']}",
    "tipo": "{row['tipo']}"
  }}
]
"""
    
    try:
        response = client.models.generate_content(
            model="gemini-3-pro-preview",  
            config={"response_mime_type": "application/json"},
            contents=[prompt]
        )
        
        variations = json.loads(response.text)
        
        if len(variations) != 8:
            print(f"Fila {idx}: generó {len(variations)} variaciones en lugar de 8")
        
        all_variations.extend(variations)
        
        time.sleep(1)  
        
    except Exception as e:
        print(f" Error en fila {idx}: {e}")
        continue


df_synthetic = pd.DataFrame(all_variations)
df_synthetic.to_csv('../data/dataset_sintetico.csv', index=False)

In [63]:
df_final.shape

(60, 3)

In [58]:
print(response.text)

[
  {
    "texto": "EMETRA: Notificacion de remision vencida por Q250. Evita el bloqueo de tu vehiculo pagando hoy aqui: https://emetra-pago.icu",
    "etiqueta": "spam",
    "categoria": "Aviso gubernamental (SAT, IGSS, Renap, Migración)"
  },
  {
    "texto": "SAT: Tu omiso de vehiculos genero una multa de Q500. Regulariza tu situacion antes de 24 hrs o habra recargos. Ingresa a: http://sat-portal.xyz",
    "etiqueta": "spam",
    "categoria": "Aviso gubernamental (SAT, IGSS, Renap, Migración)"
  },
  {
    "texto": "Migracion GT: Su pasaporte tiene un problema en el sistema. Para no perder su cita, valide sus datos y pague Q150 de tramite extra en: https://migracion-gob.info",
    "etiqueta": "spam",
    "categoria": "Aviso gubernamental (SAT, IGSS, Renap, Migración)"
  },
  {
    "texto": "Renap: Te informamos que tu DPI esta proximo a ser desactivado por falta de actualizacion. Cancela la multa de Q300 en este link: bit.ly/renap-gt",
    "etiqueta": "spam",
    "categoria": "Aviso

In [51]:
print(response.text)


```json
[
  {
    "texto": "¡ALERTA Banrural! Se realizó un cargo de Q2846.00 a tu cuenta. Si no fuiste vos, actuá URGENTE en https://gt-banca-u1s4e.icu/xrt para evitar el bloqueo.",
    "etiqueta": "spam",
    "categoria": "Financiero (préstamos, inversiones, criptomonedas, pólizas, cheques)"
  },
  {
    "texto": "Hemos detectado una transacción inusual de Q3705.00 en tu tarjeta BAM. Confirma en https://gt-banca-e5k6g.top/e5t o tu cuenta será suspendida.",
    "etiqueta": "spam",
    "categoria": "Financiero (préstamos, inversiones, criptomonedas, pólizas, cheques)"
  },
  {
    "texto": "URGENTE: Tu cuenta Tigo Money presenta un saldo negativo de Q2025.00 por cargos no autorizados. ¡Regulá tu situación aquí: https://banca-k5l6p.xyz/p9r!",
    "etiqueta": "spam",
    "categoria": "Financiero (préstamos, inversiones, criptomonedas, pólizas, cheques)"
  },
  {
    "texto": "Notificación Banrural: Tu acceso a banca en línea será suspendido. Realiza la verificación de seguridad en https:

In [59]:
data = json.loads(response.text)

In [60]:
df_genrated = pd.DataFrame(data)

In [61]:
df_genrated.head()

,texto,etiqueta,categoria
0,EMETRA: Notificacion de remision vencida por Q...,spam,"Aviso gubernamental (SAT, IGSS, Renap, Migración)"
1,SAT: Tu omiso de vehiculos genero una multa de...,spam,"Aviso gubernamental (SAT, IGSS, Renap, Migración)"
2,Migracion GT: Su pasaporte tiene un problema e...,spam,"Aviso gubernamental (SAT, IGSS, Renap, Migración)"
3,Renap: Te informamos que tu DPI esta proximo a...,spam,"Aviso gubernamental (SAT, IGSS, Renap, Migración)"
4,"IGSS: Estimado afiliado, tienes un saldo pendi...",spam,"Aviso gubernamental (SAT, IGSS, Renap, Migración)"


In [62]:
df_genrated.shape

(80, 3)